# 🌆 Delhi Deep Dive — Module 00: Exploratory Data Analysis
## Multi-Hazard Climate Risk for 11 Delhi Districts (2010–2023)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
import os

OUTPUT_DIR = '../data/outputs/delhi'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DELHI_DISTRICTS = {
    'Central Delhi':    {'lat': 28.6508, 'lon': 77.2219},
    'East Delhi':       {'lat': 28.6271, 'lon': 77.2907},
    'New Delhi':        {'lat': 28.6139, 'lon': 77.2090},
    'North Delhi':      {'lat': 28.7041, 'lon': 77.1025},
    'North East Delhi': {'lat': 28.6921, 'lon': 77.2979},
    'North West Delhi': {'lat': 28.7219, 'lon': 77.0878},
    'Shahdara':         {'lat': 28.6735, 'lon': 77.2894},
    'South Delhi':      {'lat': 28.5244, 'lon': 77.1855},
    'South East Delhi': {'lat': 28.5677, 'lon': 77.2965},
    'South West Delhi': {'lat': 28.5672, 'lon': 77.0699},
    'West Delhi':       {'lat': 28.6541, 'lon': 77.0832},
}
print(f'Delhi districts: {len(DELHI_DISTRICTS)}')
print('Years: 2010–2023 (14 years)')
print('Hazards: Drought | Flood | Heatwave | Air Quality | Water Scarcity')

## Fetch Sample Data — Central Delhi Climate Overview

In [ ]:
import time

def fetch_archive(lat, lon, variables, start='2022-01-01', end='2022-12-31'):
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': variables, 'timezone': 'Asia/Kolkata'
    }
    r = requests.get('https://archive-api.open-meteo.com/v1/archive', params=params)
    r.raise_for_status()
    return r.json()['daily']

# Fetch for Central Delhi
data = fetch_archive(28.6508, 77.2219,
    ['precipitation_sum', 'temperature_2m_max', 'et0_fao_evapotranspiration'])
df = pd.DataFrame({
    'date': pd.to_datetime(data['time']),
    'rainfall_mm': data['precipitation_sum'],
    'temp_max_c': data['temperature_2m_max'],
    'et0_mm': data['et0_fao_evapotranspiration']
})
df['month'] = df['date'].dt.month
print(f'Fetched {len(df)} daily records for Central Delhi 2022')
print(df.describe().round(2))

## Climate Overview Charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Central Delhi — 2022 Climate Overview', fontsize=14, fontweight='bold')

# 1. Monthly rainfall
monthly_rain = df.groupby('month')['rainfall_mm'].sum()
axes[0,0].bar(monthly_rain.index, monthly_rain.values, color='steelblue', alpha=0.8)
axes[0,0].set_title('Monthly Rainfall (mm)')
axes[0,0].set_xlabel('Month')
axes[0,0].set_xticks(range(1,13))
axes[0,0].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[0,0].grid(axis='y', alpha=0.4)

# 2. Monthly max temperature
monthly_temp = df.groupby('month')['temp_max_c'].mean()
axes[0,1].plot(monthly_temp.index, monthly_temp.values, 'o-', color='tomato', linewidth=2)
axes[0,1].axhline(40, color='red', linestyle='--', alpha=0.6, label='Heatwave threshold (40°C)')
axes[0,1].set_title('Monthly Avg Max Temperature (°C)')
axes[0,1].set_xlabel('Month')
axes[0,1].set_xticks(range(1,13))
axes[0,1].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[0,1].legend(fontsize=8)
axes[0,1].grid(alpha=0.4)

# 3. Rainfall vs ET0
monthly_et0 = df.groupby('month')['et0_mm'].sum()
width = 0.35
x = np.arange(12)
axes[1,0].bar(x - width/2, monthly_rain.values, width, label='Rainfall', color='steelblue', alpha=0.8)
axes[1,0].bar(x + width/2, monthly_et0.values, width, label='ET₀', color='orange', alpha=0.8)
axes[1,0].set_title('Rainfall vs Evapotranspiration (mm)')
axes[1,0].set_xlabel('Month')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[1,0].legend()
axes[1,0].grid(axis='y', alpha=0.4)

# 4. Water deficit (ET0 - rain, clipped to 0)
deficit = (monthly_et0 - monthly_rain).clip(lower=0)
axes[1,1].fill_between(deficit.index, deficit.values, alpha=0.6, color='crimson', label='Water deficit')
axes[1,1].set_title('Monthly Water Deficit (ET₀ − Rainfall, mm)')
axes[1,1].set_xlabel('Month')
axes[1,1].set_xticks(range(1,13))
axes[1,1].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[1,1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/delhi_eda_overview.png', dpi=150)
plt.show()
print('Chart saved to data/outputs/delhi/delhi_eda_overview.png')

## District Map — Hazard Context

In [ ]:
YAMUNA_PROXIMITY = {
    'North East Delhi': 1.0, 'East Delhi': 0.9, 'Shahdara': 0.9,
    'North Delhi': 0.7, 'Central Delhi': 0.5, 'New Delhi': 0.3,
    'South East Delhi': 0.4, 'South Delhi': 0.2, 'West Delhi': 0.1,
    'South West Delhi': 0.1, 'North West Delhi': 0.2,
}
GROUNDWATER_STRESS = {
    'South Delhi': 1.0, 'South West Delhi': 0.95, 'West Delhi': 0.90,
    'North West Delhi': 0.85, 'Central Delhi': 0.80, 'New Delhi': 0.75,
    'South East Delhi': 0.85, 'North Delhi': 0.60, 'East Delhi': 0.55,
    'Shahdara': 0.50, 'North East Delhi': 0.45,
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Delhi Districts — Hazard Context', fontsize=13, fontweight='bold')

for district, coords in DELHI_DISTRICTS.items():
    prox = YAMUNA_PROXIMITY[district]
    ax1.scatter(coords['lon'], coords['lat'], s=prox*500+100,
                c=prox, cmap='Blues', vmin=0, vmax=1, alpha=0.85, edgecolors='navy', linewidth=0.8)
    ax1.annotate(district.replace(' Delhi','').replace(' ',  '\n'),
                 (coords['lon'], coords['lat']), ha='center', va='bottom',
                 fontsize=7, fontweight='bold')
ax1.set_title('Yamuna Flood Proximity Score')
ax1.set_xlabel('Longitude'); ax1.set_ylabel('Latitude')
ax1.grid(alpha=0.3)

for district, coords in DELHI_DISTRICTS.items():
    stress = GROUNDWATER_STRESS[district]
    ax2.scatter(coords['lon'], coords['lat'], s=stress*500+100,
                c=stress, cmap='Reds', vmin=0, vmax=1, alpha=0.85, edgecolors='darkred', linewidth=0.8)
    ax2.annotate(district.replace(' Delhi','').replace(' ',  '\n'),
                 (coords['lon'], coords['lat']), ha='center', va='bottom',
                 fontsize=7, fontweight='bold')
ax2.set_title('Groundwater Stress Index (1.0=Critical)')
ax2.set_xlabel('Longitude')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/delhi_district_context.png', dpi=150)
plt.show()
print('Saved delhi_district_context.png')

## Summary

This EDA notebook establishes the climate context for all 11 Delhi districts. Key findings:

- **Flood risk**: North East Delhi, East Delhi and Shahdara are directly on the Yamuna floodplain
- **Groundwater**: South and South West Delhi are critically over-exploited
- **Heatwave**: All districts face similar heat stress — Delhi's compact geography means minimal variation
- **Air Quality**: The PM2.5 crisis affects all districts but industrial corridors in the north-east are most impacted

Proceed to Module 01 onwards for quantitative hazard modelling.